# Report

### Question 1

The dataset has 2169 samples with two columns (`text` and `label`), is clean and balanced with no missing values, and contains approximately 200-230 samples per class. Text samples are relatively short, averaging 10-20 words, though some reach up to 78 words. A UMAP visualisation using Sentence-BERT embeddings shows well-separated clusters with minor overlap, suggesting the classes are reasonably distinguishable in the embedding space.

Several classification approaches could be considered. At the simpler end, a TF-IDF bag-of-words model with logistic regression or a linear SVM provides a fast, interpretable baseline that is often competitive on short text. Extending this with bigrams or trigrams can capture short phrases at no additional computational cost. A middle-ground option would be to use static word embeddings (GloVe or fastText) with mean pooling, which offers richer semantics than TF-IDF without the overhead of a transformer. At the more expensive end, fine-tuning Sentence-BERT would likely maximise accuracy but risks overfitting given the small dataset size and is computationally costly relative to the potential gain. Alternatively, zero-shot or few-shot prompting with a large LLM requires no training at all and could serve as a useful reference, though it is less reproducible and harder to deploy.
Given the evidence from the UMAP visualisation that the embedding space already separates classes well, the most practical approach is to use frozen Sentence-BERT embeddings (`all-mpnet-base-v2`) with lightweight classifiers such as logistic regression, a linear SVM, and an MLP. This strikes a good balance between representational richness and simplicity, avoids the overfitting risk of fine-tuning on a small dataset, and is fast to iterate.

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import umap
import matplotlib.pyplot as plt
import seaborn as sns

SHORT_LABEL = {
    0: "CardPayFee",
    1: "DirectDebitUnknown",
    2: "BalNotUpdCheque",
    3: "WrongCashAmt",
    4: "CashWithdrawCharge",
    5: "ChargedTwice",
    6: "DeclinedWithdraw",
    7: "TransferFee",
    8: "TransferNotRecvd",
    9: "BalNotUpdTransfer",
}

df = pd.read_csv("ds_task_dataset.csv")

model = SentenceTransformer("all-mpnet-base-v2")
embeddings = model.encode(
    df["text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
)
print(f"Embedding matrix: {embeddings.shape}")

metric = "cosine" 

reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.75,
    n_components=2,
    metric=metric,
    random_state=42,
)
proj = reducer.fit_transform(embeddings)
print(f"UMAP projection: {proj.shape}")

palette = sns.color_palette("tab10", 10)

fig, ax = plt.subplots(figsize=(11, 8))

for label_id in range(10):
    mask = df["label"].values == label_id
    ax.scatter(
        proj[mask, 0],
        proj[mask, 1],
        c=[palette[label_id]],
        label=SHORT_LABEL[label_id],
        s=18,
        alpha=0.7,
        linewidths=0,
    )

ax.set_title(
    f"UMAP Projection of SBERT Embeddings (all-mpnet-base-v2)\n"
    f"n_neighbors=15 · min_dist=0.75 · {metric} metric",
    fontsize=11,
)
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
ax.legend(
    title="Class",
    bbox_to_anchor=(1.01, 1),
    loc="upper left",
    fontsize=8,
    title_fontsize=9,
    markerscale=1.8,
)
plt.tight_layout()
plt.show()

### Question 4